# Deteksi Tuberkulosis dari Citra X-ray — Pipeline v2
### Untuk paper J. ICT Res. Appl. — 5 model + optimasi threshold

**Yang baru di versi ini:**
- Menjalankan **5 arsitektur** sekaligus untuk perbandingan.
- **Optimasi threshold** dari validation set untuk memperbaiki recall pada kelas TB (mengatasi masalah false-negative yang tinggi pada threshold default 0.5).
- Melaporkan metrik pada **threshold default (0.5) DAN threshold teroptimasi**, sehingga Anda punya bahan tabel perbandingan yang kaya untuk paper.

**WAJIB:** aktifkan GPU (Runtime > Change runtime type > T4 GPU), lalu Runtime > Run all.


## 1. Cek GPU

In [ ]:
!nvidia-smi -L

## 2. Unduh dataset dari Kaggle (publik, tanpa token)

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
!unzip -q -o tuberculosis-tb-chest-xray-dataset.zip -d data_raw
print("Selesai unduh & ekstrak.")

## 3. Tata folder

In [ ]:
import shutil, os
src = "data_raw/TB_Chest_Radiography_Database"
os.makedirs("data/main", exist_ok=True)
for cls in ["Normal", "Tuberculosis"]:
    dst = f"data/main/{cls}"
    if not os.path.isdir(dst):
        shutil.move(f"{src}/{cls}", dst)
print("Isi data/main:", os.listdir("data/main"))
print("Normal:", len(os.listdir("data/main/Normal")))
print("Tuberculosis:", len(os.listdir("data/main/Tuberculosis")))

## 4. Konfigurasi & import
**Sekarang `MODELS_TO_RUN` sudah berisi kelima model.** Dengan GPU T4, total butuh beberapa jam. Jika sesi Colab gratis terputus, jalankan per-batch (mis. 2-3 model dulu), hasil tiap model tetap tersimpan.

In [ ]:
import os, time, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import (
    ResNet50, EfficientNetB0, MobileNetV2, DenseNet121, VGG16,
    resnet50, efficientnet, mobilenet_v2, densenet, vgg16)
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, accuracy_score, f1_score)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD, EPOCHS_FT = 8, 12
LR_HEAD, LR_FT = 1e-3, 1e-5
SEED = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15
OUT_DIR = "outputs"
DATA_MAIN = "data/main"
DATA_EXTERNAL = None

# Kelima model sekaligus
MODELS_TO_RUN = ["ResNet50","EfficientNetB0","MobileNetV2","DenseNet121","VGG16"]

ALL_MODELS = {
    "ResNet50":       (ResNet50,       resnet50.preprocess_input,     "conv5_block3_out"),
    "EfficientNetB0": (EfficientNetB0, efficientnet.preprocess_input, "top_conv"),
    "MobileNetV2":    (MobileNetV2,    mobilenet_v2.preprocess_input, "out_relu"),
    "DenseNet121":    (DenseNet121,    densenet.preprocess_input,     "relu"),
    "VGG16":          (VGG16,          vgg16.preprocess_input,        "block5_conv3"),
}
tf.random.set_seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

## 5. Fungsi (data, model, evaluasi, optimasi threshold, plot, Grad-CAM)

In [ ]:
def build_datasets(preprocess):
    full = tf.keras.utils.image_dataset_from_directory(
        DATA_MAIN, labels="inferred", label_mode="binary",
        class_names=["Normal","Tuberculosis"], image_size=IMG_SIZE,
        batch_size=None, shuffle=True, seed=SEED)
    n = full.cardinality().numpy()
    n_test, n_val = int(n*TEST_SPLIT), int(n*VAL_SPLIT)
    test_ds = full.take(n_test); rest = full.skip(n_test)
    val_ds = rest.take(n_val);   train_ds = rest.skip(n_val)
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomZoom(0.10), layers.RandomContrast(0.10)])
    AT = tf.data.AUTOTUNE
    def prep(img, lab, tr):
        img = preprocess(tf.cast(img, tf.float32))
        if tr: img = aug(img, training=True)
        return img, lab
    train_ds = train_ds.map(lambda x,y: prep(x,y,True), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    val_ds   = val_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    test_ds  = test_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    return train_ds, val_ds, test_ds

def build_model(builder):
    base = builder(include_top=False, weights="imagenet",
                   input_shape=IMG_SIZE+(3,), pooling="avg")
    base.trainable = False
    inp = tf.keras.Input(shape=IMG_SIZE+(3,))
    x = base(inp, training=False)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out), base

def get_probs(model, ds):
    yt, yp = [], []
    for xb, yb in ds:
        yp.extend(model.predict(xb, verbose=0).ravel().tolist())
        yt.extend(yb.numpy().ravel().tolist())
    return np.array(yt), np.array(yp)

def metrics_at(yt, yp, thr):
    pred = (yp >= thr).astype(int)
    acc = accuracy_score(yt, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
    fpr, tpr, _ = roc_curve(yt, yp); a = auc(fpr, tpr)
    return {"threshold":round(float(thr),3),"accuracy":acc,"precision":pr,"recall":rc,"f1":f1,"auc":a}

def best_threshold(yt, yp):
    """Cari threshold yang memaksimalkan F1 pada validation set."""
    grid = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(yt, (yp>=t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])

def measure_inference(model, n=50):
    d = tf.random.normal((1,)+IMG_SIZE+(3,)); model.predict(d, verbose=0)
    t0=time.time()
    for _ in range(n): model.predict(d, verbose=0)
    return (time.time()-t0)/n*1000

def plot_history(name,h1,h2):
    acc=h1.history["accuracy"]+h2.history["accuracy"]
    val=h1.history["val_accuracy"]+h2.history["val_accuracy"]
    plt.figure(figsize=(6,4)); plt.plot(acc,label="train"); plt.plot(val,label="val")
    plt.axvline(len(h1.history["accuracy"])-0.5, ls="--", c="gray")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"History {name}"); plt.legend()
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/history_{name}.png", dpi=150); plt.close()

def plot_cm(name, yt, yp, thr, tag):
    cm = confusion_matrix(yt, (yp>=thr).astype(int))
    plt.figure(figsize=(4.5,4)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
    plt.title(f"Confusion Matrix {name} ({tag}, thr={thr:.2f})")
    plt.xticks([0,1],["Normal","TB"]); plt.yticks([0,1],["Normal","TB"])
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(cm[i,j]),ha="center",va="center",
                     color="white" if cm[i,j]>cm.max()/2 else "black")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cm_{name}_{tag}.png", dpi=150); plt.close()

def gradcam(model, preprocess, name, last_conv, sample):
    if not sample or not os.path.isfile(sample): return
    img = tf.keras.utils.load_img(sample, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    x = preprocess(np.expand_dims(arr.copy(),0))
    base = model.layers[1]
    gm = models.Model(base.inputs, [base.get_layer(last_conv).output, base.output])
    with tf.GradientTape() as tape:
        conv, feat = gm(x); tape.watch(conv)
        h = feat
        for L in model.layers[2:]: h = L(h)
        pred = h
    grads = tape.gradient(pred, conv)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv = conv[0]
    heat = tf.squeeze(conv @ pooled[...,None])
    heat = tf.maximum(heat,0)/(tf.reduce_max(heat)+1e-8)
    heat = heat.numpy()
    up = np.kron(heat, np.ones((IMG_SIZE[0]//heat.shape[0]+1, IMG_SIZE[1]//heat.shape[1]+1)))[:IMG_SIZE[0],:IMG_SIZE[1]]
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1); plt.imshow(arr.astype("uint8")); plt.axis("off"); plt.title("Input")
    plt.subplot(1,2,2); plt.imshow(arr.astype("uint8")); plt.imshow(up, cmap="jet", alpha=0.45)
    plt.axis("off"); plt.title(f"Grad-CAM {name}")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/gradcam_{name}.png", dpi=150); plt.close()

def find_tb_sample():
    d = os.path.join(DATA_MAIN,"Tuberculosis")
    for f in os.listdir(d):
        if f.lower().endswith((".png",".jpg",".jpeg")): return os.path.join(d,f)
    return ""
print("Semua fungsi siap.")

## 6. Training, evaluasi & optimasi threshold
Untuk tiap model:
1. Latih dua tahap (head lalu fine-tune) dengan class weight.
2. Cari threshold optimal dari **validation set** (memaksimalkan F1).
3. Laporkan metrik test pada threshold **default (0.5)** dan **teroptimasi**.
4. Simpan confusion matrix kedua threshold + Grad-CAM + history.

In [ ]:
rows_default, rows_tuned, roc_data, tradeoff = [], [], {}, []
sample = find_tb_sample()
class_weight = {0: 1.0, 1: 3500/700}

for name in MODELS_TO_RUN:
    builder, preprocess, last_conv = ALL_MODELS[name]
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    train_ds, val_ds, test_ds = build_datasets(preprocess)
    model, base = build_model(builder)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)

    model.compile(optimizers.Adam(LR_HEAD), "binary_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                   callbacks=[es], class_weight=class_weight, verbose=2)
    base.trainable = True
    for L in base.layers[:-30]: L.trainable = False
    model.compile(optimizers.Adam(LR_FT), "binary_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                   callbacks=[es], class_weight=class_weight, verbose=2)
    plot_history(name, h1, h2)

    # cari threshold optimal dari VALIDATION (jujur, bukan dari test)
    yv, pv = get_probs(model, val_ds)
    thr = best_threshold(yv, pv)

    # evaluasi TEST pada dua threshold
    yt, yp = get_probs(model, test_ds)
    m_def = metrics_at(yt, yp, 0.5)
    m_tun = metrics_at(yt, yp, thr)
    rows_default.append({"model":name, **{k:round(v,4) for k,v in m_def.items()}})
    rows_tuned.append({"model":name,  **{k:round(v,4) for k,v in m_tun.items()}})
    roc_data[name] = (yt, yp)
    plot_cm(name, yt, yp, 0.5, "default")
    plot_cm(name, yt, yp, thr, "tuned")
    tradeoff.append({"model":name, "accuracy":round(m_def["accuracy"],4),
                     "params_millions":round(model.count_params()/1e6,2),
                     "inference_ms":round(measure_inference(model),2)})
    try: gradcam(model, preprocess, name, last_conv, sample)
    except Exception as e: print("Grad-CAM gagal:", e)

    print(f"  DEFAULT thr=0.50 -> acc={m_def['accuracy']:.4f} recall(TB)={m_def['recall']:.4f} f1={m_def['f1']:.4f}")
    print(f"  TUNED   thr={thr:.2f} -> acc={m_tun['accuracy']:.4f} recall(TB)={m_tun['recall']:.4f} f1={m_tun['f1']:.4f}")
    print(f"  AUC={m_def['auc']:.4f}")
    tf.keras.backend.clear_session()

print("\nSelesai semua model.")

## 7. Simpan tabel & ROC

In [ ]:
df_def = pd.DataFrame(rows_default); df_tun = pd.DataFrame(rows_tuned)
df_def.to_csv(f"{OUT_DIR}/results_default_threshold.csv", index=False)
df_tun.to_csv(f"{OUT_DIR}/results_tuned_threshold.csv", index=False)
pd.DataFrame(tradeoff).to_csv(f"{OUT_DIR}/tradeoff.csv", index=False)

plt.figure(figsize=(6,5))
for name,(yt,yp) in roc_data.items():
    fpr,tpr,_ = roc_curve(yt,yp); plt.plot(fpr,tpr,label=f"{name} (AUC={auc(fpr,tpr):.3f})")
plt.plot([0,1],[0,1],"k--",alpha=0.4)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Comparison"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/roc_comparison.png", dpi=150); plt.show()

print("=== Threshold default (0.5) ==="); display(df_def)
print("=== Threshold teroptimasi (max F1 dari validation) ==="); display(df_tun)
print("=== Trade-off ukuran/kecepatan ==="); display(pd.DataFrame(tradeoff))

## 8. Lihat visual (confusion matrix default vs tuned, Grad-CAM, history)

In [ ]:
from IPython.display import Image, display
for name in MODELS_TO_RUN:
    for fn in [f"cm_{name}_default", f"cm_{name}_tuned", f"gradcam_{name}", f"history_{name}"]:
        p = f"{OUT_DIR}/{fn}.png"
        if os.path.isfile(p):
            print(f"--- {fn} ---"); display(Image(p))

## 9. Simpan ke Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
dest = "/content/drive/MyDrive/tb_outputs_v2"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree("outputs", dest)
print("Tersimpan ke:", dest)

---
### Cara membaca hasil untuk paper

**Dua tabel utama:**
- `results_default_threshold.csv` — performa pada threshold 0.5 (cara konvensional).
- `results_tuned_threshold.csv` — performa setelah threshold dioptimasi untuk F1.

Bandingkan keduanya: threshold teroptimasi akan **menaikkan recall TB** (lebih sedikit pasien terlewat) dengan sedikit penurunan precision. Inilah trade-off yang tepat untuk skrining medis, dan menjadi salah satu poin diskusi terkuat di paper Anda.

**Untuk klaim kuat (opsional tapi disarankan reviewer):** ulangi seluruh proses dengan beberapa SEED berbeda (mis. 42, 1, 7, 123, 2024) lalu laporkan mean ± std. Selisih kecil antar model bisa jadi sekadar noise.
